# **Tools & Tool Binding**
Tools give an LLM the ability to take actions — search the web, query a database, etc.

`llm.bind_tools(tools)` attaches tool schemas to every LLM call so the model can request a tool when needed.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get('GROQ_API_KEY'):
    print("Groq API Key is set.")
else:
    raise ValueError("GROQ_API_KEY not found.")

In [ ]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

### **DuckDuckGo Search Tool**
`@tool` turns any function into a LangChain-compatible tool.
The docstring becomes the tool's description — the LLM reads it to decide when to call the tool.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def search_duckduckgo(query: str) -> str:
    """This tool searches the latest news on DuckDuckGo for the given query and returns the results."""
    duck_search = DuckDuckGoSearchRun()
    return duck_search.invoke(query)

### **Arxiv Query Tool**

In [ ]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

@tool
def arxiv_tool(query: str) -> str:
    """This tool allows you to query the arXiv database for research papers."""
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(query)

### **Wikipedia Search Tool**

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(query: str):
    """This tool allows you to search Wikipedia for information on a given topic."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

### **Custom Tool** — a static lookup table

In [ ]:
@tool
def personal_info(name: str):
    """Use this tool to get personal information about Alice, Bob, or Charlie."""
    info = {
        "Alice":   "Alice is a software engineer with 5 years of experience in AI.",
        "Bob":     "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups."
    }
    return info.get(name, "No information available for this person.")

### **Tool Binding**
`bind_tools()` attaches the tool schemas to the LLM.
When queried, the LLM can now return a `tool_calls` list instead of (or in addition to) a text response.

In [ ]:
tools = [search_duckduckgo, arxiv_tool, wiki_tool, personal_info]

# bind_tools injects the tool schemas so the LLM knows what tools are available
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# The LLM decides it needs to call a tool and returns tool_calls instead of plain text
response = llm_with_tools.invoke("What is the latest news on AI?")

# tool_calls is a list of dicts: [{"name": "search_duckduckgo", "args": {...}, "id": "..."}]
print(response.tool_calls)